# Daily Challenge — Build a Tiny Agent with Tools

**Week 10 · Day 3 — Bootcamp GenAI & Machine Learning 2026**

Build a beginner agent using **smolagents** that can call two custom tools — no paid API keys required.

| Tool | What it does |
|------|--------------|
| `KBLookupTool` | Searches an in-memory knowledge base for matching snippets |
| `MathTool` | Adds or multiplies two numbers |

```
Query
  └──► ToolCallingAgent
              │
              ├──► KBLookupTool  → [kb:N] snippet
              └──► MathTool      → result
                        ↓
                  Final answer (cited)
```

---
## Step 1 — Install

In [ ]:
!pip install -q "smolagents[transformers]" wikipedia

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Toggle: set True to load a real HF model (needs more RAM/time)
USE_REAL_MODEL = False

print(f"Model mode: {'TransformersModel (real)' if USE_REAL_MODEL else 'Manual orchestration (stub)'}")

---
## Step 2 — Knowledge Base (5–8 snippets)

In [ ]:
KB_SNIPPETS = [
    {"id": 1, "text": "An agentic AI loop is a cycle where an agent observes its environment, plans an action, executes a tool, and then observes the result — repeating until the goal is met."},
    {"id": 2, "text": "smolagents is a lightweight Python library by Hugging Face for building tool-calling agents. It supports CodeAgent (generates Python code) and ToolCallingAgent (structured function calls)."},
    {"id": 3, "text": "RAG (Retrieval-Augmented Generation) grounds LLM answers in external documents, reducing hallucination by fetching relevant context before generating a response."},
    {"id": 4, "text": "A Tool in smolagents is a Python class that subclasses Tool and implements a forward() method. Each tool declares its name, description, input types, and output type."},
    {"id": 5, "text": "MCP (Model Context Protocol) by Anthropic standardizes how AI models connect to external tools and resources over STDIO or HTTP transports."},
    {"id": 6, "text": "LangChain is an open-source framework providing abstractions for LLM chains, agents, and retrievers. LCEL uses the | operator to compose pipelines."},
    {"id": 7, "text": "Vector stores like FAISS store text embeddings for semantic similarity search. They are the backbone of RAG pipelines: queries are embedded, nearest neighbors fetched as context."},
    {"id": 8, "text": "A ToolCallingAgent uses structured JSON tool calls. The model outputs {tool_name, arguments}, the framework executes the tool, appends the result, and continues until a final answer is produced."},
]

print(f"KB loaded ✅ — {len(KB_SNIPPETS)} snippets")
for s in KB_SNIPPETS:
    print(f"  [kb:{s['id']}] {s['text'][:60]}...")

---
## Step 3 — Implement KBLookupTool

In [ ]:
from smolagents import Tool


class KBLookupTool(Tool):
    name = "kb_lookup"
    description = (
        "Search the in-memory knowledge base for snippets matching the query keywords. "
        "Returns matching snippets with source tags like [kb:N]. "
        "Use this when the question is about AI, agents, RAG, LangChain, MCP, or smolagents."
    )
    inputs = {
        "query": {
            "type": "string",
            "description": "Keywords or a short question to search for in the knowledge base."
        }
    }
    output_type = "string"

    def forward(self, query: str) -> str:
        keywords = query.lower().split()
        matches = [
            s for s in KB_SNIPPETS
            if any(kw in s["text"].lower() for kw in keywords)
        ]
        if not matches:
            return (
                f"No KB entries found for '{query}'. "
                "I don't have evidence on this topic. "
                "Suggested follow-up: try a different keyword or check Wikipedia."
            )
        # Return top 3 matches with citations
        return "\n".join(
            f"[kb:{s['id']}] {s['text']}"
            for s in matches[:3]
        )


kb_tool = KBLookupTool()

# Test
print("=== KBLookupTool test ===")
print(kb_tool.forward("agentic AI loop"))

---
## Step 4 — Implement MathTool

In [ ]:
class MathTool(Tool):
    name = "math_tool"
    description = (
        "Perform basic arithmetic on two numbers. "
        "Use op='add' for addition, op='multiply' for multiplication."
    )
    inputs = {
        "a": {"type": "number", "description": "First number"},
        "b": {"type": "number", "description": "Second number"},
        "op": {
            "type": "string",
            "description": "Operation to perform: 'add' or 'multiply'"
        }
    }
    output_type = "string"

    def forward(self, a: float, b: float, op: str) -> str:
        op_lower = op.lower().strip()
        if op_lower in {"add", "+", "plus", "sum"}:
            result = a + b
            return f"{a} + {b} = {result}"
        elif op_lower in {"multiply", "*", "mul", "times", "product"}:
            result = a * b
            return f"{a} × {b} = {result}"
        else:
            return f"Unknown operation '{op}'. Use 'add' or 'multiply'."


math_tool = MathTool()

# Tests
print("=== MathTool tests ===")
print(math_tool.forward(12, 30, "add"))
print(math_tool.forward(7, 6, "multiply"))
print(math_tool.forward(5, 3, "unknown"))

---
## Step 5 — Wire the Agent

### Option A — TransformersModel (real local model)
> Note: `tiny-gpt2` lacks tool-calling capability. For real tool calls, use a model like `Qwen/Qwen2.5-0.5B-Instruct`.

### Option B — Manual orchestration (default, always works)
> Simulates the agent loop: parse query → select tool → execute → format answer.

In [ ]:
tools = [kb_tool, math_tool]

if USE_REAL_MODEL:
    from smolagents import ToolCallingAgent, TransformersModel

    model = TransformersModel(
        model_id="Qwen/Qwen2.5-0.5B-Instruct",
        device_map="auto"
    )
    agent = ToolCallingAgent(tools=tools, model=model, max_steps=3)
    print("Real ToolCallingAgent ready ✅")

else:
    print("Using manual orchestration (stub mode) ✅")
    print(f"Tools registered: {[t.name for t in tools]}")

In [ ]:
import re


def manual_agent(query: str) -> dict:
    """
    Manual agentic loop — simulates ToolCallingAgent behavior:
    1. Parse intent from query
    2. Select and call the right tool
    3. Format final answer with source citations
    """
    q_lower = query.lower()

    # --- Math intent detection ---
    math_keywords = ["add", "sum", "plus", "multiply", "times", "product", "+", "×", "*"]
    is_math = any(kw in q_lower for kw in math_keywords)

    if is_math:
        numbers = re.findall(r'\d+(?:\.\d+)?', query)
        a = float(numbers[0]) if len(numbers) > 0 else 0
        b = float(numbers[1]) if len(numbers) > 1 else 0
        op = "multiply" if any(k in q_lower for k in ["multiply", "times", "product", "*"]) else "add"

        tool_call = {"tool": "math_tool", "args": {"a": a, "b": b, "op": op}}
        tool_result = math_tool.forward(a, b, op)

        return {
            "query": query,
            "tool_calls": [tool_call],
            "answer": f"The result is: **{tool_result}**"
        }

    # --- KB lookup for everything else ---
    tool_call = {"tool": "kb_lookup", "args": {"query": query}}
    tool_result = kb_tool.forward(query)

    if "No KB entries" in tool_result:
        answer = (
            f"I don't have evidence on this topic in my knowledge base. "
            f"Suggested follow-up: try searching Wikipedia or rephrasing your query."
        )
    else:
        # Extract first match as the answer
        first_result = tool_result.split("\n")[0]
        # Isolate citation and text
        citation_match = re.match(r'(\[kb:\d+\])', first_result)
        citation = citation_match.group(1) if citation_match else ""
        text = first_result.replace(citation, "").strip()[:300]
        answer = f"{text} {citation}"

    return {
        "query": query,
        "tool_calls": [tool_call],
        "answer": answer
    }


def run_agent(query: str) -> dict:
    """Unified runner — uses real agent or manual orchestration."""
    if USE_REAL_MODEL:
        result_text = agent.run(query)
        return {"query": query, "tool_calls": "(see agent logs)", "answer": result_text}
    return manual_agent(query)


print("Agent runner defined ✅")

---
## Step 6 — Run 3 Test Queries

In [ ]:
TEST_QUERIES = [
    "Add 12 and 30.",
    "Multiply 7 by 6.",
    "What is an agentic AI loop?",
]

sep = "=" * 60

for query in TEST_QUERIES:
    result = run_agent(query)
    print(sep)
    print(f"QUERY      : {result['query']}")
    print(f"TOOL CALLS : {result['tool_calls']}")
    print(f"ANSWER     : {result['answer']}")
    print()

---
## Bonus — Additional Queries

In [ ]:
BONUS_QUERIES = [
    "What is smolagents?",
    "How does a ToolCallingAgent work?",
    "What is the capital of Mars?",  # no evidence → graceful fallback
]

for query in BONUS_QUERIES:
    result = run_agent(query)
    print(sep)
    print(f"QUERY      : {result['query']}")
    print(f"TOOL CALLS : {result['tool_calls']}")
    print(f"ANSWER     : {result['answer']}")
    print()

---
## Summary

| Step | Built | Status |
|------|-------|--------|
| 1 | `pip install smolagents[transformers] wikipedia` | ✅ |
| 2 | 8-snippet KB with source IDs | ✅ |
| 3 | `KBLookupTool` — keyword match, returns `[kb:N]` citations | ✅ |
| 4 | `MathTool` — add/multiply with op parameter | ✅ |
| 5 | `ToolCallingAgent` (real) or manual orchestration (stub) | ✅ |
| 6 | 3 queries: Add 12+30, Multiply 7×6, agentic AI loop | ✅ |

### Tool Architecture

```python
class MyTool(Tool):
    name = "my_tool"          # identifier used by the agent
    description = "..."       # tells the LLM WHEN to use this tool
    inputs = {"arg": {...}}   # typed parameters
    output_type = "string"    # return type

    def forward(self, arg: str) -> str:
        return ...            # the actual tool logic
```

### Agentic Loop (ToolCallingAgent)

```
1. Agent receives query
2. LLM decides: which tool? with what args?
3. Framework calls tool.forward(**args)
4. Result appended to conversation
5. LLM generates final answer (or calls another tool)
6. Loop exits when max_steps reached or final answer produced
```

### Citation Convention

- `[kb:N]` — snippet N from the in-memory knowledge base
- `[wiki:Title]` — Wikipedia article (for external lookups)
- No evidence → explicit message + suggested follow-up (no hallucination)